In [1]:
%pip install matplotlib


In [2]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data_loading import load_transactions
from src.preprocessing import clean_transactions, save_cleaned_data

# Load raw data
df_raw = load_transactions("../data/raw/transactions.csv")

# Clean data
df_clean = clean_transactions(df_raw)

# Save cleaned data
save_cleaned_data(df_clean, "../data/processed/cleaned_transactions.csv")

df_clean.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   CustomerID          1000 non-null   int64         
 1   PurchaseDate        1000 non-null   datetime64[ns]
 2   TransactionAmount   1000 non-null   float64       
 3   ProductInformation  1000 non-null   object        
 4   OrderID             1000 non-null   int64         
 5   Location            1000 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(2)
memory usage: 54.7+ KB


In [3]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.data_loading import load_transactions
from src.preprocessing import clean_transactions
from src.rfm_calculation import calculate_rfm

# Load and clean data
df_raw = load_transactions("../data/raw/transactions.csv")
df_clean = clean_transactions(df_raw)

# Calculate RFM
rfm_df = calculate_rfm(df_clean)

rfm_df.head()
rfm_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 946 entries, 0 to 945
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CustomerID  946 non-null    int64  
 1   Recency     946 non-null    int64  
 2   Frequency   946 non-null    int64  
 3   Monetary    946 non-null    float64
dtypes: float64(1), int64(3)
memory usage: 29.7 KB


In [4]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.rfm_calculation import calculate_rfm
from src.scoring import score_rfm

rfm_df = calculate_rfm(df_clean)

rfm_scored = score_rfm(rfm_df)

rfm_scored.head()
rfm_scored.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 946 entries, 0 to 945
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   CustomerID  946 non-null    int64  
 1   Recency     946 non-null    int64  
 2   Frequency   946 non-null    int64  
 3   Monetary    946 non-null    float64
 4   R_Score     946 non-null    int32  
 5   F_Score     946 non-null    int32  
 6   M_Score     946 non-null    int32  
dtypes: float64(1), int32(3), int64(3)
memory usage: 40.8 KB


In [5]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.segmentation import label_customers

rfm_labeled = label_customers(rfm_scored)

# Save final table
rfm_labeled.to_csv("../data/processed/rfm_table.csv", index=False)

rfm_labeled["Segment"].value_counts()
rfm_labeled.head()


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Total,Segment
0,1011,34,2,1129.02,2,4,4,10,A
1,1025,22,1,359.29,3,2,2,7,C
2,1029,1,1,704.99,4,2,3,9,A
3,1046,44,1,859.82,2,2,4,8,A
4,1049,14,1,225.72,4,2,1,7,C


In [6]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.visualization import create_all_plots

create_all_plots(
    rfm_labeled,
    "../outputs/charts"
)


In [7]:
from pathlib import Path

def write_rfm_insights(df, output_path):
    total_customers = len(df)

    segment_counts = df["Segment"].value_counts()
    segment_percent = (segment_counts / total_customers * 100).round(2)

    avg_metrics = df.groupby("Segment")[["Recency", "Frequency", "Monetary"]].mean().round(2)

    lines = []
    lines.append("RFM CUSTOMER SEGMENTATION – BUSINESS INSIGHTS\n")
    lines.append(f"Total Customers Analyzed: {total_customers}\n")

    lines.append("Customer Distribution by Segment:\n")
    for seg in segment_counts.index:
        lines.append(
            f"- Segment {seg}: {segment_counts[seg]} customers ({segment_percent[seg]}%)"
        )

    lines.append("\nAverage RFM Metrics by Segment:\n")
    lines.append(avg_metrics.to_string())

    lines.append("\n\nKey Insights:\n")
    lines.append("- Segment A customers are the most valuable and recently active.")
    lines.append("- Segment B customers show moderate engagement and can be nurtured.")
    lines.append("- Segment C customers are low value or inactive and may need re-engagement campaigns.")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w") as f:
        f.write("\n".join(lines))


In [8]:
write_rfm_insights(
    rfm_labeled,
    "../outputs/reports/rfm_insights.txt"
)


In [9]:
import os

required_outputs = [
    "../data/processed/cleaned_transactions.csv",
    "../data/processed/rfm_table.csv",
    "../outputs/charts/recency_distribution.png",
    "../outputs/charts/frequency_distribution.png",
    "../outputs/charts/monetary_distribution.png",
    "../outputs/charts/segment_counts.png",
    "../outputs/reports/rfm_insights.txt",
]

missing = [f for f in required_outputs if not os.path.exists(f)]

if not missing:
    print("✅ FINAL CHECK PASSED: All outputs generated successfully.")
else:
    print("❌ Missing files:")
    for f in missing:
        print(f)


✅ FINAL CHECK PASSED: All outputs generated successfully.


In [10]:
import pandas as pd

# Total revenue
total_revenue = rfm_labeled["Monetary"].sum()

# Revenue contribution table
revenue_summary = (
    rfm_labeled
    .groupby("Segment")
    .agg(
        Customers=("CustomerID", "count"),
        Revenue=("Monetary", "sum")
    )
    .reset_index()
)

revenue_summary["Revenue %"] = (
    revenue_summary["Revenue"] / total_revenue * 100
).round(2)

revenue_summary


,Segment,Customers,Revenue,Revenue %
0,A,381,294128.45,57.26
1,C,565,219549.36,42.74
